In [ ]:
# ── Starter Cell 1: Load environment variables ──────────────────────────────
from dotenv import load_dotenv   # imports load_dotenv to read the .env file
import os                         # imports os so we can read environment variables

load_dotenv()                     # reads .env in the current directory and sets env vars

api_key = os.getenv("ANTHROPIC_API_KEY")  # retrieves the key; returns None if missing

print("API key loaded:", api_key is not None)  # True = key found; False = check your .env

In [ ]:
# ── Starter Cell 2: Initialise the Anthropic client ─────────────────────────
from anthropic import Anthropic   # imports the Anthropic SDK client class

client = Anthropic(api_key=api_key)  # creates a reusable client; api_key auth is passed explicitly

print("Anthropic client ready")   # confirms the object was created without errors

In [ ]:
# ── Starter Cell 3: Smoke test — verify end-to-end connectivity ──────────────
response = client.messages.create(
    model="claude-haiku-4-5",          # fast, cheap model — perfect for smoke tests
    #    model="claude-sonnet-4-5",    # uncomment for higher-quality responses
    max_tokens=50,                     # hard ceiling; Claude stops here even mid-sentence
    messages=[{"role": "user", "content": "Say hello in one short sentence."}]  # minimal prompt
)

print(response.content[0].text)       # .content is a list of blocks; [0] gets the first text block

# CCA Lab: Making a Request to the Claude API

**Course:** Building with the Claude API  
**Sub-module:** Foundations  
**Lesson:** Making a Request

---

## What this lab covers
- Setting up your environment with `python-dotenv` and the Anthropic SDK
- Understanding `client.messages.create()` parameters: `model`, `max_tokens`, `messages`
- Extracting text from the response object (`response.content[0].text`)
- Using the `system` parameter to shape Claude's behaviour
- Building multi-turn conversations by accumulating the `messages` list
- Writing, scoring, and iteratively improving prompts with an objective rubric
- Identifying and recovering from common failure modes
- Tracking token usage across an entire notebook session

## CCA domains exercised
- API mechanics & request structure
- Prompt engineering & iterative improvement
- Evaluation & grading
- Responsible API usage (cost / token awareness)

## Learning objectives
1. Construct a valid `client.messages.create()` call from scratch
2. Explain why `response.content` is a list and when it contains more than one block
3. Distinguish what belongs in `system` vs. `messages`
4. Build a multi-turn conversation by manually accumulating the `messages` list
5. Score prompts with a numeric rubric and iterate until the threshold is met
6. Name at least six failure modes and their production fixes
7. Read the `usage` object to monitor per-call and session-level token costs

## Section 1: Prerequisites and Environment Setup
## CCA objective demonstrated: Configuring a reproducible API environment with dotenv

In [ ]:
# Verify required packages are importable — run this before anything else
import sys

required = ["anthropic", "dotenv"]   # packages the rest of the notebook depends on

for pkg in required:                  # iterate over each required package name
    try:
        __import__(pkg)               # attempt to import by name string
        print(f"✅ {pkg}")            # success — package is available
    except ImportError:
        print(f"❌ {pkg} — run: pip install {pkg}")  # tell the user exactly what to install

print(f"\nPython {sys.version}")      # log the Python version for reproducibility

## Section 2: API Glossary and Reference
## CCA objective demonstrated: Mapping every `messages.create()` parameter to its purpose and constraints

| Parameter | Type | Required | Description | Common mistake |
|-----------|------|----------|-------------|----------------|
| `model` | `str` | ✅ | Model identifier, e.g. `"claude-haiku-4-5"` | Typo in model string → 404 |
| `max_tokens` | `int` | ✅ | Hard output ceiling in tokens | Omitting it → `ValidationError` |
| `messages` | `list[dict]` | ✅ | Alternating `user`/`assistant` turns | Wrong role order → API error |
| `system` | `str` | ❌ | Persistent persona / constraint block | Putting persona in user turn |
| `temperature` | `float 0–1` | ❌ | Randomness; 0 = deterministic | Setting to 0 for creative tasks |
| `stop_sequences` | `list[str]` | ❌ | Strings that halt generation | Not knowing stop truncates output |

**Response object fields used in this lab**

| Field | What it contains |
|-------|------------------|
| `response.content` | List of content blocks (text, tool_use, …) |
| `response.content[0].text` | The text of the first (usually only) block |
| `response.usage.input_tokens` | Tokens consumed by the prompt |
| `response.usage.output_tokens` | Tokens in Claude's reply |
| `response.stop_reason` | Why generation stopped: `end_turn`, `max_tokens`, `stop_sequence` |
| `response.model` | The model that actually served the request |

## Section 3: Client Setup and `ask()` Helper
## CCA objective demonstrated: Encapsulating `messages.create()` in a reusable, annotated helper with usage logging

In [ ]:
# ── Usage log — accumulates every API call made in this notebook session ─────
usage_log = []   # plain list; each entry is a dict with label, tokens, stop_reason, etc.

def log_usage(response, label=""):
    """
    Append a token-usage record to the session log.

    Parameters
    ----------
    response : anthropic.types.Message
        The raw response object returned by client.messages.create().
    label : str
        Human-readable tag identifying which call this was (e.g. 'smoke_test').
    """
    usage_log.append({              # add a new record to the running log
        "label": label,             # caller-supplied tag so we can identify the call later
        "model": response.model,    # which model actually served the request
        "input_tokens": response.usage.input_tokens,    # tokens the prompt consumed
        "output_tokens": response.usage.output_tokens,  # tokens in Claude's reply
        "stop_reason": response.stop_reason,            # why generation stopped
    })

print("usage_log initialised and log_usage() defined")

In [ ]:
def ask(prompt, system=None, model="claude-haiku-4-5", max_tokens=256, label="ask"):
    """
    Send a single-turn request to Claude and return the response text.

    Parameters
    ----------
    prompt     : str   — the user's question or instruction
    system     : str   — optional system prompt (persona / constraints)
    model      : str   — Claude model identifier
    max_tokens : int   — hard output ceiling
    label      : str   — tag for the usage log

    Returns
    -------
    str  — the text of Claude's first content block
    """
    # Build the keyword-argument dict dynamically so we only pass 'system'
    # when the caller supplied one — passing system=None would still be valid
    # but being explicit keeps the call clean and avoids accidental overrides.
    kwargs = {
        "model": model,             # which Claude model to use
        "max_tokens": max_tokens,   # REQUIRED — omitting this raises ValidationError
        "messages": [{"role": "user", "content": prompt}],  # single-turn conversation
    }

    if system is not None:          # only add the 'system' key when a value was provided
        kwargs["system"] = system   # system is a TOP-LEVEL kwarg, not inside messages[]

    response = client.messages.create(**kwargs)  # unpack the dict as keyword args

    # response.content is a LIST because a single reply can contain multiple blocks:
    # e.g. a text block followed by a tool_use block in function-calling workflows.
    # For standard text completions there is exactly one block, so [0] is safe.
    text = response.content[0].text   # [0] = first block; .text = the string content

    # stop_reason tells us WHY generation ended:
    #   'end_turn'      — Claude finished naturally (ideal)
    #   'max_tokens'    — hit our ceiling; output is TRUNCATED (potential bug!)
    #   'stop_sequence' — a stop string was matched
    if response.stop_reason == "max_tokens":
        print(f"⚠️  [{label}] stop_reason=max_tokens — response may be truncated!")

    log_usage(response, label=label)  # record tokens + stop_reason before returning

    return text   # caller gets just the string, not the full response object


# Quick verification
reply = ask("What is 2 + 2? Answer in one word.", label="section3_test")
print("ask() returned:", reply)

## Section 4: Intentional Error Demonstration
## CCA objective demonstrated: Recognising and diagnosing `ValidationError` from missing required parameters

In [ ]:
# Deliberately omit max_tokens to trigger a ValidationError.
# This is the most common beginner mistake — the API REQUIRES max_tokens.

print("Attempting a call WITHOUT max_tokens ...\n")

try:
    bad_response = client.messages.create(
        model="claude-haiku-4-5",
        # max_tokens intentionally omitted
        messages=[{"role": "user", "content": "Hello"}]
    )
except Exception as e:               # catch any SDK or API error
    print(f"Error type : {type(e).__name__}")   # shows the specific exception class
    print(f"Error detail: {e}")                  # human-readable explanation
    print("\n✅ Expected — always supply max_tokens in every messages.create() call.")

print("\nOther common errors:")
print("  AuthenticationError  — wrong or missing ANTHROPIC_API_KEY")
print("  BadRequestError      — malformed messages list (wrong role order, empty content)")
print("  RateLimitError       — too many requests per minute for your tier")

## Section 5: The `system` Parameter
## CCA objective demonstrated: Using the system prompt to encode persistent persona and constraints, separate from user intent

### When and why to use `system`
The `system` parameter is processed **before** the conversation begins. Claude treats it as standing instructions that persist across every turn. It is the right place for anything that should be true for the **entire session**, regardless of what the user says.

### Decision table: `system` vs. `messages`

| Belongs in `system` | Belongs in `messages` (user turn) |
|---------------------|-----------------------------------|
| Persona: "You are a concise physics tutor" | The actual question: "What is a qubit?" |
| Output format rules: "Always respond in bullet points" | Context the user is providing this turn |
| Constraints: "Never discuss competitor products" | Follow-up clarifications |
| Tone: "Use plain English, no jargon" | Examples the user wants Claude to process |
| Scope: "Only answer questions about quantum computing" | Uploaded documents or data for this request |

**Architectural principle:** The `system` parameter encodes *who Claude is* and *how it must behave*; the user turn encodes *what the user wants right now*.

In [ ]:
QUESTION = "What is quantum computing?"   # same question used in both calls for fair comparison

SYSTEM_PROMPT = (
    "You are a concise physics tutor for undergraduate students. "
    "Always respond in plain English with no jargon. "
    "Keep every answer under 60 words."
)

# ── Call 1: WITHOUT system prompt ────────────────────────────────────────────
reply_no_system = ask(QUESTION, label="no_system")   # uses ask() defaults (no system)

# ── Call 2: WITH system prompt ───────────────────────────────────────────────
reply_with_system = ask(QUESTION, system=SYSTEM_PROMPT, label="with_system")  # persona applied

print("=" * 60)
print("WITHOUT system prompt:")
print(reply_no_system)
print("\n" + "=" * 60)
print("WITH system prompt (concise physics tutor):")
print(reply_with_system)
print("=" * 60)
print("\nObservation: the system prompt enforces brevity and plain language")
print("without requiring the user to repeat those constraints every turn.")

## Section 5b: Multi-Turn Conversation — Messages List Accumulation
## CCA objective demonstrated: Building stateful dialogue by manually appending every turn to the `messages` list

The Claude API is **stateless** — it has no memory between calls. To simulate memory, we must
send the entire conversation history on every request. The `messages` list grows with each turn:
each new request includes all prior user *and* assistant turns.

> **Architectural insight:** The **first user message** should clearly establish the topic because
> all subsequent context is built on top of it. A vague opener forces Claude to infer intent
> across all subsequent turns, degrading coherence for the entire conversation.

In [ ]:
def chat(messages, system=None, model="claude-haiku-4-5", max_tokens=300, label="chat"):
    """
    Send the full conversation history to Claude and return the reply text.

    Internally calls client.messages.create() directly — no extra abstraction layer.
    The caller is responsible for appending both the user turn AND this reply
    to `messages` before the next call.

    Parameters
    ----------
    messages   : list[dict]  — accumulated conversation history
    system     : str         — optional system prompt
    model      : str         — Claude model identifier
    max_tokens : int         — hard output ceiling
    label      : str         — tag for the usage log

    Returns
    -------
    str  — Claude's reply text
    """
    kwargs = {
        "model": model,           # which Claude model to call
        "max_tokens": max_tokens, # required ceiling
        "messages": messages,     # the FULL history — this is what enables multi-turn memory
    }
    if system is not None:        # conditionally add system prompt
        kwargs["system"] = system

    # ── This is client.messages.create() being called directly ───────────────
    response = client.messages.create(**kwargs)   # raw SDK call — no extra wrapper
    # ─────────────────────────────────────────────────────────────────────────

    log_usage(response, label=label)              # record tokens and stop_reason
    return response.content[0].text               # return plain text to caller


print("chat() helper defined — calls client.messages.create() directly")

In [ ]:
# ── Three-turn conversation demonstrating history accumulation ────────────────
messages = []   # start with an empty history — the API requires this to be a list

# ── Turn 1 ───────────────────────────────────────────────────────────────────
user_1 = "What is a qubit in quantum computing?"
messages.append({"role": "user", "content": user_1})   # append user turn BEFORE calling
reply_1 = chat(messages, label="multi_turn_1")          # send full history (1 message)
messages.append({"role": "assistant", "content": reply_1})  # append assistant reply

# ── Turn 2 ───────────────────────────────────────────────────────────────────
user_2 = "How does superposition relate to that?"
messages.append({"role": "user", "content": user_2})   # history now has 3 entries
reply_2 = chat(messages, label="multi_turn_2")          # sends ALL 3 prior messages
messages.append({"role": "assistant", "content": reply_2})

# ── Turn 3 ───────────────────────────────────────────────────────────────────
user_3 = "Can you give me a real-world analogy for that concept?"
messages.append({"role": "user", "content": user_3})   # history now has 5 entries
reply_3 = chat(messages, label="multi_turn_3")          # sends ALL 5 prior messages
messages.append({"role": "assistant", "content": reply_3})

print(f"Turn 1 → {reply_1[:120]}...\n")
print(f"Turn 2 → {reply_2[:120]}...\n")
print(f"Turn 3 → {reply_3[:120]}...\n")
print(f"Final messages list length: {len(messages)} entries (3 user + 3 assistant)")

In [ ]:
# ── Token cost-of-history analysis ───────────────────────────────────────────
# Each turn sends the FULL history, so input tokens grow with every turn.
# This is the core cost driver in multi-turn applications.

# Extract the multi-turn entries from usage_log
mt_entries = [e for e in usage_log if e["label"] in ("multi_turn_1", "multi_turn_2", "multi_turn_3")]

print("Token cost of history accumulation:")
print(f"{'Turn':<12} {'Input tokens':>14} {'Output tokens':>14} {'Stop reason':>14}")
print("-" * 58)

for entry in mt_entries:                     # iterate in order
    turn_num = entry["label"].split("_")[-1] # extract '1', '2', or '3'
    print(
        f"Turn {turn_num:<7} "
        f"{entry['input_tokens']:>14} "
        f"{entry['output_tokens']:>14} "
        f"{entry['stop_reason']:>14}"
    )

if len(mt_entries) == 3:                     # only compute delta if all 3 turns ran
    delta = mt_entries[2]["input_tokens"] - mt_entries[0]["input_tokens"]
    hypothetical_single = mt_entries[0]["input_tokens"]  # cost if it were a single call
    print(f"\nInput token growth Turn 1 → Turn 3: +{delta} tokens")
    print(f"Extra context cost vs. single-turn: +{delta} input tokens")
    print("\n💡 Architectural implication: in long conversations, history compression")
    print("   (summarisation or sliding window) is essential for cost control.")

## Section 6: Rubric Scoring Function
## CCA objective demonstrated: Operationalising output quality into a deterministic, numeric rubric with per-criterion feedback

> **Grader reliability note:** Deterministic rubrics can over-score responses that
> contain the right words but wrong semantics. For production evals, complement
> keyword checks with a Claude-as-judge semantic pass — the rule + judge layering
> pattern from the evaluation labs.

In [ ]:
import re   # regular expressions for pattern-based grading

THRESHOLD = 3   # minimum score (out of 4) considered a PASS

def score_response(text, verbose=True):
    """
    Score a quantum-computing explanation on four independent criteria.

    The rubric is DETERMINISTIC — the same input always produces the same score.
    Each criterion contributes exactly 1 point; total range is 0–4.
    The function also enforces a pass/fail verdict against THRESHOLD.

    Parameters
    ----------
    text    : str  — Claude's response to evaluate
    verbose : bool — if True, print per-criterion results with ✅/❌ icons

    Returns
    -------
    int  — total score (0–4)
    """
    results = {}   # will hold {criterion_name: (score_int, description)}

    # ── Criterion 1: Conciseness (≤80 words) ─────────────────────────────────
    # .split() with no argument splits on ANY whitespace — a fast word-count proxy.
    # It's not perfect (contractions count as one word) but is consistent and fast.
    word_count = len(text.split())
    c1 = int(word_count <= 80)   # int(bool) converts True→1, False→0
    results["Conciseness (≤80 words)"] = (c1, f"{word_count} words")

    # ── Criterion 2: Defines 'qubit' ──────────────────────────────────────────
    # re.search() scans the ENTIRE string (unlike re.match which anchors to the start).
    # \b is a word boundary — ensures we match the whole word 'qubit', not 'qubits'
    # embedded in another token. re.IGNORECASE handles capitalisation variants.
    c2 = int(bool(re.search(r"\bqubit\b", text, re.IGNORECASE)))
    results["Defines 'qubit'"] = (c2, "keyword found" if c2 else "keyword missing")

    # ── Criterion 3: Mentions superposition OR entanglement ───────────────────
    # The pipe | inside the regex means logical OR — either keyword satisfies the criterion.
    c3 = int(bool(re.search(r"\b(superposition|entanglement)\b", text, re.IGNORECASE)))
    results["Mentions superposition/entanglement"] = (c3, "found" if c3 else "missing")

    # ── Criterion 4: Contrasts with classical computing ───────────────────────
    # We search for 'classical' as a whole word. This ensures 'neoclassical' doesn't
    # accidentally satisfy the criterion.
    c4 = int(bool(re.search(r"\bclassical\b", text, re.IGNORECASE)))
    results["Contrasts with classical computing"] = (c4, "found" if c4 else "missing")

    total = c1 + c2 + c3 + c4   # simple integer sum; each criterion worth 1 point

    if verbose:
        print("── Rubric Scorecard ──────────────────────────────────")
        for criterion, (score, detail) in results.items():
            icon = "✅" if score else "❌"   # visual pass/fail indicator per criterion
            # Print each dimension individually with its numeric contribution
            print(f"  {icon} {criterion}: +{score}  ({detail})")
        print(f"── Total: {total}/4 ", end="")

        # Self-contained pass/fail verdict — the rubric function enforces the threshold
        if total >= THRESHOLD:
            print(f"✅ PASS (≥{THRESHOLD}/4)")
        else:
            print(f"❌ FAIL (<{THRESHOLD}/4)")
        print("──────────────────────────────────────────────────────")

    return total


# Demonstrate with a sample text
sample = "A qubit differs from a classical bit by exploiting superposition."
score_response(sample)

## Section 7: Improvement Cycle — Write → Evaluate → Improve → Re-evaluate
## CCA objective demonstrated: Applying the iterative prompt-engineering loop with objective numeric evidence

> **Grader reliability note:** Deterministic rubrics can over-score responses that
> contain the right words but wrong semantics. For production evals, complement
> keyword checks with a Claude-as-judge semantic pass — the rule + judge layering
> pattern from the evaluation labs.

In [ ]:
# ── Version 1: Vague prompt ───────────────────────────────────────────────────
PROMPT_V1 = "Explain quantum computing."

reply_v1 = ask(PROMPT_V1, label="improve_v1")   # call the API; log usage automatically
score_v1 = score_response(reply_v1)              # score with verbose=True (default)

print()

# ── Version 2: Specific, constrained prompt ───────────────────────────────────
# Changes made vs. V1:
#   1. Added explicit word limit ("in 60 words or fewer") → targets Criterion 1
#   2. Required definition of 'qubit'                    → targets Criterion 2
#   3. Required mention of superposition/entanglement     → targets Criterion 3
#   4. Required contrast with classical computing         → targets Criterion 4
PROMPT_V2 = (
    "Explain quantum computing in 60 words or fewer. "
    "Your answer MUST: (1) define what a qubit is, "
    "(2) mention superposition or entanglement, and "
    "(3) contrast quantum computing with classical computing."
)

reply_v2 = ask(PROMPT_V2, label="improve_v2")   # call the API; log usage automatically
score_v2 = score_response(reply_v2)              # score with verbose=True (default)

In [ ]:
# ── Conditional V3: only generate if V2 failed ────────────────────────────────
# This demonstrates that the improvement loop CAN repeat — it's not just a one-shot pass.

best_score = max(score_v1, score_v2)   # track the best score achieved so far

if best_score < THRESHOLD:
    print(f"❌ FAIL — best score so far ({best_score}/{4}) is below threshold ({THRESHOLD}).")
    print("   Iterating to v3 ...\n")

    # V3 adds an even stronger constraint: provide the answer in a numbered list
    PROMPT_V3 = (
        "You are a physics tutor. Respond in exactly three numbered sentences. "
        "Sentence 1: define qubit. "
        "Sentence 2: describe superposition or entanglement. "
        "Sentence 3: contrast with classical computing. "
        "Total length must be 80 words or fewer."
    )
    reply_v3 = ask(PROMPT_V3, label="improve_v3")
    score_v3 = score_response(reply_v3)
    best_score = max(best_score, score_v3)
    versions = [
        ("V1", PROMPT_V1, reply_v1, score_v1),
        ("V2", PROMPT_V2, reply_v2, score_v2),
        ("V3", PROMPT_V3, reply_v3, score_v3),
    ]
else:
    print(f"✅ V2 passed (score={score_v2}/{4} ≥ threshold={THRESHOLD}) — no V3 needed.\n")
    versions = [
        ("V1", PROMPT_V1, reply_v1, score_v1),
        ("V2", PROMPT_V2, reply_v2, score_v2),
    ]

# ── Per-criterion side-by-side comparison table ───────────────────────────────
# We re-score silently (verbose=False) to collect per-criterion data for the table
import re   # ensure re is available (already imported in Section 6)

def score_breakdown(text):
    """Return a dict of {criterion_short_name: score_int} for table display."""
    wc  = int(len(text.split()) <= 80)
    qb  = int(bool(re.search(r"\bqubit\b", text, re.IGNORECASE)))
    sup = int(bool(re.search(r"\b(superposition|entanglement)\b", text, re.IGNORECASE)))
    cls = int(bool(re.search(r"\bclassical\b", text, re.IGNORECASE)))
    return {"≤80 words": wc, "qubit": qb, "super/entangle": sup, "classical": cls}

criteria = ["≤80 words", "qubit", "super/entangle", "classical"]

# ── Print side-by-side comparison table ──────────────────────────────────────
print("\n" + "=" * 72)
print("SIDE-BY-SIDE COMPARISON")
print("=" * 72)
# Header row
header = f"{'Version':<8} {'Prompt (truncated)':<32} " + " ".join(f"{c:>14}" for c in criteria) + f" {'TOTAL':>7}"
print(header)
print("-" * 72)

for ver, prompt, reply, total in versions:
    bd = score_breakdown(reply)                     # per-criterion scores
    prompt_short = (prompt[:29] + "...") if len(prompt) > 29 else prompt.ljust(32)
    criteria_cols = " ".join(f"{'✅' if bd[c] else '❌':>14}" for c in criteria)
    print(f"{ver:<8} {prompt_short:<32} {criteria_cols} {total:>7}")

print("-" * 72)

# Conditional PASS/FAIL line
if best_score >= THRESHOLD:
    print(f"\n✅ PASS — best score {best_score}/4 meets threshold {THRESHOLD}/4")
else:
    print(f"\n❌ FAIL — best score {best_score}/4 is below threshold {THRESHOLD}/4 — iterate further")

## Section 8: Failure Mode Analysis
## CCA objective demonstrated: Cataloguing and live-demonstrating API failure modes as production reliability bugs

| # | Failure mode | Root cause | Symptom | Production fix |
|---|-------------|------------|---------|----------------|
| 1 | `max_tokens` too low | Output ceiling too tight | Reply truncated mid-sentence | Profile typical response length; set ceiling 2× median |
| 2 | Missing `role` key | Malformed message dict | `BadRequestError` | Validate message dicts with a helper before sending |
| 3 | Wrong message order | assistant turn before user | `BadRequestError` | Always start with `{"role": "user", ...}` |
| 4 | `temperature=0` for creative tasks | Collapsed output distribution | Repetitive, formulaic responses | Use temperature 0.7–1.0 for creative; 0 for factual |
| 5 | Empty `content` string | No instruction given | Unpredictable or empty response | Validate `len(content.strip()) > 0` before calling |
| 6 | Vague prompt → high output variance | Under-specified instruction | Non-deterministic application behaviour | Add explicit format and content constraints to the prompt |

**Frame:** Output variance from under-specified prompts is a **reliability bug**, not an aesthetic preference — it means your application behaviour is non-deterministic across runs.

In [ ]:
# ── Live demo 1: max_tokens truncation ───────────────────────────────────────
print("=" * 60)
print("FAILURE MODE 1: max_tokens too low → truncated output")
print("=" * 60)

truncated = ask(
    "Write a detailed explanation of quantum computing with examples.",
    max_tokens=15,    # intentionally tiny ceiling to force truncation
    label="truncation_demo"
)
print("Response:", truncated)

# Check stop_reason from log
last_entry = usage_log[-1]
print(f"\nstop_reason: {last_entry['stop_reason']}")    # should be 'max_tokens'
print(f"output_tokens: {last_entry['output_tokens']}")
print("\n✅ Diagnosis: stop_reason='max_tokens' confirms truncation.")
print("   Fix: increase max_tokens or shorten the prompt to leave room for the reply.")

In [ ]:
# ── Live demo 2: Vague prompt → output variance (run same prompt 3×) ──────────
# This demonstrates that under-specified prompts produce non-deterministic output —
# a reliability bug in production, not merely an aesthetic concern.

print("=" * 60)
print("FAILURE MODE 6: Vague prompt → high output variance")
print("=" * 60)

VAGUE_PROMPT = "Tell me about computers."   # under-specified: no length, topic, or format

word_counts = []      # collect word counts for each run
responses_vague = []  # collect response texts

for i in range(3):   # run the same vague prompt exactly 3 times
    r = ask(VAGUE_PROMPT, max_tokens=200, label=f"variance_run_{i+1}")
    wc = len(r.split())           # word-count proxy: split on whitespace
    word_counts.append(wc)        # accumulate for range calculation
    responses_vague.append(r)     # store full text for inspection
    print(f"Run {i+1}: {wc} words | first 80 chars: {r[:80].replace(chr(10), ' ')}...")

wc_range = max(word_counts) - min(word_counts)   # range = max - min
print(f"\nWord counts: {word_counts}")
print(f"Range (max - min): {wc_range} words")
print(f"\n⚠️  A range of {wc_range} words means your app will behave differently each run.")
print("   This is a RELIABILITY BUG. Fix: add explicit length and format constraints.")

## Section 9: Token Usage Analysis
## CCA objective demonstrated: Reading `response.usage` to monitor per-call and session-level token costs, including stop_reason and output variance across prompt versions

In [ ]:
# ── Session-wide token usage report ──────────────────────────────────────────
print("SESSION TOKEN USAGE REPORT")
print("=" * 80)
print(f"{'Label':<22} {'Model':<20} {'Input':>8} {'Output':>8} {'Total':>8} {'Stop':>12}")
print("-" * 80)

total_input = 0
total_output = 0

for entry in usage_log:                          # iterate every recorded API call
    inp  = entry["input_tokens"]                 # tokens the prompt consumed
    out  = entry["output_tokens"]                # tokens in Claude's reply
    tot  = inp + out                             # combined cost of this call
    stop = entry.get("stop_reason", "n/a")       # why generation stopped
    mdl  = entry["model"][-12:]                  # truncate model name for display
    total_input  += inp                          # accumulate session totals
    total_output += out
    print(f"{entry['label']:<22} {mdl:<20} {inp:>8} {out:>8} {tot:>8} {stop:>12}")

print("=" * 80)
print(f"{'SESSION TOTALS':<22} {'':>20} {total_input:>8} {total_output:>8} {total_input+total_output:>8}")
print()

# ── Output token variance: V1 vs V2 ──────────────────────────────────────────
# Demonstrates how adding explicit constraints to the prompt affects output length (and cost).
v1_entry = next((e for e in usage_log if e["label"] == "improve_v1"), None)
v2_entry = next((e for e in usage_log if e["label"] == "improve_v2"), None)

if v1_entry and v2_entry:
    v1_out = v1_entry["output_tokens"]
    v2_out = v2_entry["output_tokens"]
    delta  = v2_out - v1_out                 # negative = V2 used fewer output tokens
    print("── Prompt version output token comparison ─────────────────────────────")
    print(f"  V1 output tokens: {v1_out}")
    print(f"  V2 output tokens: {v2_out}")
    print(f"  Δ = {delta:+d} tokens")
    if delta < 0:
        print(f"  💡 V2's explicit constraints reduced output by {abs(delta)} tokens — lower cost.")
    elif delta > 0:
        print(f"  💡 V2 produced {delta} more tokens — constraints may have required more elaboration.")
    else:
        print("  Output length was identical across versions.")
    print("\n  Insight: explicit prompt constraints typically reduce output token variance,")
    print("  making costs more predictable and reducing truncation risk.")

## Section 10: Practice Drills
## CCA objective demonstrated: Applying lab concepts independently to new tasks

In [ ]:
# ── Drill 1: Write and score a prompt for a different topic ───────────────────
# Task: Write a prompt that asks Claude to explain machine learning in ≤60 words,
# defining 'neural network', mentioning 'training data', and contrasting with
# rule-based systems. Then score it with a similar rubric.

YOUR_PROMPT = """Explain machine learning in 60 words or fewer. 
Your answer MUST: (1) define what a neural network is, 
(2) mention training data, and 
(3) contrast machine learning with rule-based systems."""

# YOUR_PROMPT = "..."   # ← Replace with your own prompt

drill1_reply = ask(YOUR_PROMPT, label="drill1")
print("Your prompt's response:")
print(drill1_reply)
print()

# Quick rubric for ML topic
wc   = len(drill1_reply.split()) <= 60
nn   = bool(re.search(r"\bneural network\b", drill1_reply, re.IGNORECASE))
td   = bool(re.search(r"\btraining data\b", drill1_reply, re.IGNORECASE))
rb   = bool(re.search(r"\brule.based\b", drill1_reply, re.IGNORECASE))
d1_score = int(wc) + int(nn) + int(td) + int(rb)
print(f"Drill 1 score: {d1_score}/4 | wc≤60={'✅' if wc else '❌'} neural={'✅' if nn else '❌'} training={'✅' if td else '❌'} rule-based={'✅' if rb else '❌'}")

In [ ]:
# ── Drill 2: Build a 2-turn conversation with a system prompt ─────────────────
# Task: Create a conversation where Claude acts as a "Socratic tutor" (system prompt).
# Turn 1: ask a question. Turn 2: follow up based on the reply.

SOCRATIC_SYSTEM = (
    "You are a Socratic tutor. Instead of giving direct answers, "
    "respond with thought-provoking questions that guide the student to the answer. "
    "Keep each response under 40 words."
)

socratic_messages = []    # fresh conversation history

# Turn 1
q1 = "Why is the sky blue?"
socratic_messages.append({"role": "user", "content": q1})
r1 = chat(socratic_messages, system=SOCRATIC_SYSTEM, label="drill2_t1")
socratic_messages.append({"role": "assistant", "content": r1})
print(f"User: {q1}")
print(f"Socratic tutor: {r1}\n")

# Turn 2
q2 = "Hmm, is it something to do with how light scatters?"
socratic_messages.append({"role": "user", "content": q2})
r2 = chat(socratic_messages, system=SOCRATIC_SYSTEM, label="drill2_t2")
socratic_messages.append({"role": "assistant", "content": r2})
print(f"User: {q2}")
print(f"Socratic tutor: {r2}")
print(f"\nHistory length: {len(socratic_messages)} messages")

In [ ]:
# ── Drill 3: Token budget awareness ──────────────────────────────────────────
# Task: Call ask() with max_tokens=20 on a prompt that requires a longer response.
# Observe stop_reason, then fix by raising max_tokens.

print("=== Drill 3: Token budget awareness ===")

# Intentionally too-small budget
short_reply = ask(
    "List the seven continents of the world with one fact about each.",
    max_tokens=20,      # way too small for seven continents
    label="drill3_small"
)
print(f"With max_tokens=20 : '{short_reply}'")
print(f"stop_reason        : {usage_log[-1]['stop_reason']}")

# Fixed budget
full_reply = ask(
    "List the seven continents of the world with one fact about each.",
    max_tokens=300,     # generous ceiling for 7 continents
    label="drill3_full"
)
print(f"\nWith max_tokens=300: {len(full_reply.split())} words")
print(f"stop_reason        : {usage_log[-1]['stop_reason']}")
print("\n✅ Lesson: always profile a prompt's typical output length, then set")
print("   max_tokens to at least 2× that value to avoid silent truncation.")

## Section 11: CCA Takeaways
## CCA objective demonstrated: Consolidating lab concepts into exam-ready mental models

| # | Mental model | One-line rule |
|---|-------------|---------------|
| 1 | `max_tokens` is a hard ceiling | Always set it; omitting it raises `ValidationError`; too-small causes silent truncation — profile first, set to 2× median |
| 2 | `response.content` is always a list | Even for plain text, index with `[0]`; future-proofs against multi-block replies (tool_use, image blocks) |
| 3 | `system` vs. `messages` separation | System = *who Claude is* (persistent); messages = *what the user wants now* (per-request) |
| 4 | Multi-turn memory is manual | Append every user AND assistant turn to `messages`; input tokens grow with history — use compression for long sessions |
| 5 | Rubrics must be self-contained | A rubric function should print per-criterion ✅/❌, enforce a pass threshold internally, and be deterministic — complement with Claude-as-judge for semantic accuracy |
| 6 | Output variance is a reliability bug | Vague prompts produce non-deterministic output lengths and formats; explicit constraints are not style — they are correctness requirements |
| 7 | `stop_reason` is your truncation alarm | Log it on every call; `max_tokens` in `stop_reason` means your response is incomplete — treat it as an error in production |